In [ ]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torchgeo import models as tg_models
from torchgeo import datasets as tg_datasets 
from torch.utils.data import DataLoader, ChainDataset, ConcatDataset
from torch.nn import functional as F

epochs = 50
batch_size = 12

device = 'cuda'
model = tg_models.FarSeg(backbone='resnet50', classes=8, backbone_pretrained=True).to(device)
model = torch.compile(model)
train_set = tg_datasets.LoveDA(root='data', split='train', scene=['urban', 'rural'], transforms=None, download=True, checksum=False)
val_set = tg_datasets.LoveDA(root='data', split='val', scene=['urban', 'rural'], transforms=None, download=True, checksum=False)
# test_set = tg_datasets.LoveDA(root='data', split='test', scene=['urban', 'rural'], transforms=None, download=True, checksum=False)

# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8, drop_last=True, prefetch_factor=2)
# val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=True, num_workers=8, drop_last=True, prefetch_factor=2)
# test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True, num_workers=8, drop_last=True, prefetch_factor=2)


data_set = ConcatDataset([train_set, val_set])
data_loader = DataLoader(data_set, batch_size=batch_size, shuffle=True, num_workers=8, drop_last=True, prefetch_factor=2)


total_steps = epochs*len(data_set)
optimizer = AdamW(model.parameters(), lr= 1e-5)
lr_scheduler = OneCycleLR(optimizer, max_lr= 1e-3, total_steps=total_steps)
scaler = torch.amp.GradScaler()

print(len(data_loader))


import time

training_loss = []
step = 0
steps_print = 100
for epoch in range(epochs):
    for data in data_loader:
        time_start = time.time()
        image = data['image'].to(device)/255.0 #images are in range[0,255]
        mask = data['mask'].to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device, dtype=torch.float16):
            out = model(image)
            loss = F.cross_entropy(out, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        training_loss.append(loss.item())
        if step%steps_print == 0: 
            print_loss = sum(training_loss[-steps_print:])/steps_print
            print(f'epoch = {epoch} step = {step} lr = {lr_scheduler.get_last_lr()[0]:.5f} loss = {print_loss} time_per_step ={time.time()-time_start:.3f}')
        step+=1


torch.save(model.state_dict(), 'FarSeg_LoveDA.pth')


# total_steps = 209550
from matplotlib import pyplot as plt 
new_loss = torch.tensor(training_loss[:209500]).view(100, -1).mean(dim=0)

plt.plot(range(len(new_loss)), new_loss)
plt.savefig('workspace/training_loss.png')